4 Data Cleaning & Preprocessing
4.1 Drop Unnecessary Columns
Prompt: Which columns should be dropped, and why?

In your Quarto report, explain:
- Which columns are irrelevant or redundant?
- Why are we removing multiple versions of NAICS/SOC codes?
- How will this improve analysis?

4.2 Dropping Unnecessary Columns
We remove redundant NAICS/SOC codes and tracking data to simplify our dataset. Keeping only the latest NAICS_2022_6 and SOC_2021_4 ensures that our analysis reflects current industry and occupational classifications.

4.2.1 Python Implementation
columns_to_drop = [
    "ID", "URL", "ACTIVE_URLS", "DUPLICATES", "LAST_UPDATED_TIMESTAMP",
    "NAICS2", "NAICS3", "NAICS4", "NAICS5", "NAICS6",
    "SOC_2", "SOC_3", "SOC_5"
]
df.drop(columns=columns_to_drop, inplace=True)

4.3 Handle Missing Values
While performing the analysis answer the question: How should missing values be handled?

📄 Quarto Markdown Example

4.4 Handling Missing Values
We use different strategies for missing values:
- Numerical fields (e.g., Salary) are filled with the median.
- Categorical fields (e.g., Industry) are replaced with "Unknown".
- Columns with >50% missing values are dropped.

import missingno as msno

# Visualize missing data
msno.heatmap(df)
plt.title("Missing Values Heatmap")
plt.show()

# Drop columns with >50% missing values
df.dropna(thresh=len(df) * 0.5, axis=1, inplace=True)

# Fill missing values
df["Salary"].fillna(df["Salary"].median(), inplace=True)
df["Industry"].fillna("Unknown", inplace=True)

4.5 Remove Duplicates
📄 Quarto Markdown Example
## Removing Duplicate Job Postings

To ensure each job is counted only once, we remove duplicates based on job title, company, location, and posting date.

df = df.drop_duplicates(subset=["TITLE", "COMPANY", "LOCATION", "POSTED"], keep="first")

5 Exploratory Data Analysis (EDA)
In your Quarto report, explain:
- Why these visualizations were chosen.
- Key insights from each graph.

📄 Quarto Markdown Example

5.1 Job Postings by Industry
Understanding industry demand helps job seekers prioritize skill development.

5.1.1 Job Postings by Industry
fig = px.bar(df["Industry"].value_counts(), title="Job Postings by Industry")
fig.show()

5.1.2 Salary Distribution by Industry
fig = px.box(df, x="Industry", y="Salary", title="Salary Distribution by Industry")
fig.show()

5.1.3 Remote vs. On-Site Jobs
fig = px.pie(df, names="REMOTE_TYPE_NAME", title="Remote vs. On-Site Jobs")
fig.show()

6 Commit & Publish to GitHub
Ensure _quarto.yml is updated to include the “Data Analysis” tab.

quarto render
git add data_analysis.qmd _quarto.yml
git commit -m "Enhanced data cleaning, EDA, and salary outlier handling"
git push origin main

## **Data Cleaning**

In [90]:
from pyspark.sql import SparkSession

# Start a Spark session
spark = SparkSession.builder.appName("JobPostingsAnalysis").getOrCreate()

# Load the CSV file into a Spark DataFrame
df = spark.read.option("header", "true").option("inferSchema", "true").option("multiLine","true").option("escape", "\"").csv("/home/ubuntu/job-market-analysis-project-2026/data/lightcast_job_postings.csv")

# Show schema
df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- LAST_UPDATED_DATE: string (nullable = true)
 |-- LAST_UPDATED_TIMESTAMP: timestamp (nullable = true)
 |-- DUPLICATES: integer (nullable = true)
 |-- POSTED: string (nullable = true)
 |-- EXPIRED: string (nullable = true)
 |-- DURATION: integer (nullable = true)
 |-- SOURCE_TYPES: string (nullable = true)
 |-- SOURCES: string (nullable = true)
 |-- URL: string (nullable = true)
 |-- ACTIVE_URLS: string (nullable = true)
 |-- ACTIVE_SOURCES_INFO: string (nullable = true)
 |-- TITLE_RAW: string (nullable = true)
 |-- BODY: string (nullable = true)
 |-- MODELED_EXPIRED: string (nullable = true)
 |-- MODELED_DURATION: integer (nullable = true)
 |-- COMPANY: integer (nullable = true)
 |-- COMPANY_NAME: string (nullable = true)
 |-- COMPANY_RAW: string (nullable = true)
 |-- COMPANY_IS_STAFFING: boolean (nullable = true)
 |-- EDUCATION_LEVELS: string (nullable = true)
 |-- EDUCATION_LEVELS_NAME: string (nullable = true)
 |-- MIN_EDULEVELS: integer (

**Column Selection and Feature Reduction**

The original dataset contains a wide range of variables encompassing job characteristics, classification systems, metadata, and raw textual fields. To ensure analytical clarity and methodological rigor, a systematic feature selection process was undertaken to remove irrelevant, redundant, and excessively granular variables.

First, variables associated with metadata and system-level identifiers were excluded. These include fields such as ID, URL, SOURCES, and timestamp-related columns, which primarily serve operational or tracking purposes rather than contributing meaningful information for analysis. Similarly, raw text fields such as TITLE_RAW and COMPANY_RAW were removed in favor of their cleaned or standardized counterparts, such as TITLE_CLEAN, which provide greater consistency and are more suitable for downstream analysis.

Second, the dataset includes multiple occupational and industry classification systems, including NAICS, SOC, CIP, ONET, and Lightcast’s proprietary taxonomy. While each system provides a structured representation of job and industry characteristics, there is substantial conceptual overlap among them. To reduce redundancy and mitigate the risk of multicollinearity, a single, coherent set of classification variables was selected. Specifically, NAICS2_NAME was retained to represent industry at an aggregate and interpretable level, while LOT_OCCUPATION_NAME and LOT_CAREER_AREA_NAME were used to capture occupational roles and broader career domains. Other classification systems such as SOC, ONET, and CIP were excluded, as they duplicate information already captured by the selected variables without providing additional analytical value.

Furthermore, multiple levels and versions of NAICS codes, including NAICS3 through NAICS6 and NAICS_2022_*, were removed. These variables reflect a hierarchical structure in which more granular levels are nested within broader categories. Including multiple levels simultaneously would introduce redundant information and unnecessarily increase model complexity. Retaining only the 2-digit NAICS classification ensures a balance between interpretability and analytical usefulness while avoiding excessive fragmentation of categories.

Finally, highly granular and versioned taxonomy fields, such as LOT_SPECIALIZED_OCCUPATION_NAME and LOT_V6_* variables, were excluded. These fields tend to generate a large number of distinct categories, which can reduce interpretability, complicate visualization, and negatively impact model performance due to high cardinality.

Overall, this feature reduction process enhances the analytical quality of the dataset by minimizing redundancy, reducing dimensionality, and improving interpretability. By focusing on a carefully selected subset of variables, the analysis becomes more efficient, the resulting models are more stable, and the insights generated are more coherent and actionable.

**Removing Duplicates**

To ensure each job is counted only once, we remove duplicates based on job title, company, location, and posting date.

In [91]:
# select the needed columns 
cols = [
    # job roles
    "TITLE_NAME",
    "TITLE_CLEAN",
    "LOT_OCCUPATION_NAME",
    "LOT_CAREER_AREA_NAME",
    "LOT_SPECIALIZED_OCCUPATION_NAME",
    "LOT_V6_SPECIALIZED_OCCUPATION_NAME",
    "LOT_V6_CAREER_AREA_NAME",

    # skills
    "SKILLS_NAME",
    "SPECIALIZED_SKILLS_NAME",
    "SOFTWARE_SKILLS_NAME",
    "COMMON_SKILLS_NAME",
    "CERTIFICATIONS_NAME",

    # job description
    "BODY",

    # industry
    "NAICS2_NAME",
    "NAICS_2022_2_NAME",
    "LIGHTCAST_SECTORS_NAME",
    

    # experience/education
    "MIN_YEARS_EXPERIENCE",
    "MIN_EDULEVELS_NAME",
    "EDUCATION_LEVELS_NAME",
    "IS_INTERNSHIP",

    # salary
    "SALARY",
    "SALARY_FROM",
    "SALARY_TO",

    # work setup
    "REMOTE_TYPE_NAME",
    "EMPLOYMENT_TYPE_NAME"
]
    
# remove duplicate job postings
df = df.dropDuplicates(["TITLE", "COMPANY", "LOCATION", "POSTED"])

# select only relevant columns
df = df.select(*cols)

In [41]:
# print the schema and display the new df
df.printSchema()
df.show(5)

root
 |-- TITLE_NAME: string (nullable = true)
 |-- TITLE_CLEAN: string (nullable = true)
 |-- LOT_OCCUPATION_NAME: string (nullable = true)
 |-- LOT_CAREER_AREA_NAME: string (nullable = true)
 |-- SKILLS_NAME: string (nullable = true)
 |-- SPECIALIZED_SKILLS_NAME: string (nullable = true)
 |-- SOFTWARE_SKILLS_NAME: string (nullable = true)
 |-- COMMON_SKILLS_NAME: string (nullable = true)
 |-- CERTIFICATIONS_NAME: string (nullable = true)
 |-- BODY: string (nullable = true)
 |-- NAICS2_NAME: string (nullable = true)
 |-- LIGHTCAST_SECTORS_NAME: string (nullable = true)
 |-- MIN_YEARS_EXPERIENCE: integer (nullable = true)
 |-- MIN_EDULEVELS: integer (nullable = true)
 |-- IS_INTERNSHIP: boolean (nullable = true)
 |-- SALARY: integer (nullable = true)
 |-- SALARY_FROM: integer (nullable = true)
 |-- SALARY_TO: integer (nullable = true)
 |-- REMOTE_TYPE_NAME: string (nullable = true)
 |-- EMPLOYMENT_TYPE_NAME: string (nullable = true)



+----------+-----------+-------------------+--------------------+-----------+-----------------------+--------------------+------------------+-------------------+----+-----------+----------------------+--------------------+-------------+-------------+------+-----------+---------+----------------+--------------------+
|TITLE_NAME|TITLE_CLEAN|LOT_OCCUPATION_NAME|LOT_CAREER_AREA_NAME|SKILLS_NAME|SPECIALIZED_SKILLS_NAME|SOFTWARE_SKILLS_NAME|COMMON_SKILLS_NAME|CERTIFICATIONS_NAME|BODY|NAICS2_NAME|LIGHTCAST_SECTORS_NAME|MIN_YEARS_EXPERIENCE|MIN_EDULEVELS|IS_INTERNSHIP|SALARY|SALARY_FROM|SALARY_TO|REMOTE_TYPE_NAME|EMPLOYMENT_TYPE_NAME|
+----------+-----------+-------------------+--------------------+-----------+-----------------------+--------------------+------------------+-------------------+----+-----------+----------------------+--------------------+-------------+-------------+------+-----------+---------+----------------+--------------------+
|      NULL|       NULL|               NULL|  

In [94]:
# compute the percentage of missing data to assess the approach for data cleaning
from pyspark.sql.functions import col, count, when

total_rows = df.count()

missing_summary = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).toPandas().T

missing_summary.columns = ["missing_count"]
missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / total_rows * 100
)

missing_summary = missing_summary.sort_values(
    by="missing_percentage",
    ascending=False
)

missing_summary

,missing_count,missing_percentage
LIGHTCAST_SECTORS_NAME,52137,75.344663
SALARY,39960,57.747334
SALARY_FROM,38495,55.630221
SALARY_TO,38495,55.630221
MIN_YEARS_EXPERIENCE,22348,32.295731
TITLE_CLEAN,105,0.151738
TITLE_NAME,17,0.024567
LOT_CAREER_AREA_NAME,17,0.024567
LOT_OCCUPATION_NAME,17,0.024567
SPECIALIZED_SKILLS_NAME,17,0.024567


A small subset of observations (17 rows) contained missing values across nearly all relevant variables. These rows were identified as lacking meaningful information and were removed from the dataset to improve data quality and ensure the integrity of the analysis.


In [95]:
# after inspecting the table above, I noticed there are several 17 missing null values across sveral columns
# my assumption is that these null entries are the same across all columns 
# to verify this assumption I will confirm by checking rows where all these columns aree null
from pyspark.sql.functions import col

cols_to_check = [
    "LOT_OCCUPATION_NAME",
    "LOT_SPECIALIZED_OCCUPATION_NAME",
    "SKILLS_NAME",
    "SOFTWARE_SKILLS_NAME",
    "COMMON_SKILLS_NAME",
    "CERTIFICATIONS_NAME",
    "NAICS2_NAME",
    "REMOTE_TYPE_NAME",
    "EMPLOYMENT_TYPE_NAME"
]

null_rows = df.filter(
    sum([col(c).isNull().cast("int") for c in cols_to_check]) == len(cols_to_check)
)

null_rows.count()

17

In [96]:
# my verification check above returned 17 rows
# now i will visually inspect them to validate
# this will show nulls across all columns
null_rows.show(20, truncate=False)

+----------+-----------+-------------------+--------------------+-------------------------------+----------------------------------+-----------------------+-----------+-----------------------+--------------------+------------------+-------------------+----+-----------+-----------------+----------------------+--------------------+------------------+---------------------+-------------+------+-----------+---------+----------------+--------------------+
|TITLE_NAME|TITLE_CLEAN|LOT_OCCUPATION_NAME|LOT_CAREER_AREA_NAME|LOT_SPECIALIZED_OCCUPATION_NAME|LOT_V6_SPECIALIZED_OCCUPATION_NAME|LOT_V6_CAREER_AREA_NAME|SKILLS_NAME|SPECIALIZED_SKILLS_NAME|SOFTWARE_SKILLS_NAME|COMMON_SKILLS_NAME|CERTIFICATIONS_NAME|BODY|NAICS2_NAME|NAICS_2022_2_NAME|LIGHTCAST_SECTORS_NAME|MIN_YEARS_EXPERIENCE|MIN_EDULEVELS_NAME|EDUCATION_LEVELS_NAME|IS_INTERNSHIP|SALARY|SALARY_FROM|SALARY_TO|REMOTE_TYPE_NAME|EMPLOYMENT_TYPE_NAME|
+----------+-----------+-------------------+--------------------+---------------------------

In [97]:
# drop all the null rows
df = df.filter(
    sum([col(c).isNull().cast("int") for c in cols_to_check]) < len(cols_to_check)
)

### **Job Titles & Roles**


In [99]:
# view a smaller subset of selected columns
# specifically the job title related columns
df.select(
    "TITLE_NAME",
    "TITLE_CLEAN",
    "LOT_OCCUPATION_NAME",
    "LOT_CAREER_AREA_NAME",
    "LOT_SPECIALIZED_OCCUPATION_NAME",
    "LOT_V6_SPECIALIZED_OCCUPATION_NAME",
    "LOT_V6_CAREER_AREA_NAME",
).show(20, truncate=False)

+------------+------------------------------------------------------------+-------------------------------------+-------------------------------------------+--------------------------------+----------------------------------+-------------------------------------------+
|TITLE_NAME  |TITLE_CLEAN                                                 |LOT_OCCUPATION_NAME                  |LOT_CAREER_AREA_NAME                       |LOT_SPECIALIZED_OCCUPATION_NAME |LOT_V6_SPECIALIZED_OCCUPATION_NAME|LOT_V6_CAREER_AREA_NAME                    |
+------------+------------------------------------------------------------+-------------------------------------+-------------------------------------------+--------------------------------+----------------------------------+-------------------------------------------+
|Unclassified|want real freedom you need leverage                         |Business Intelligence Analyst        |Information Technology and Computer Science|Oracle Consultant / Analyst     |

In [100]:
# inspect the column
df.select("TITLE_NAME") \
  .distinct() \
  .show(50, truncate=False)

+---------------------------------------+
|TITLE_NAME                             |
+---------------------------------------+
|Project Engineering Managers           |
|Technical Subject Matter Experts       |
|Data Integrity Analysts                |
|Cloud Migration Engineers              |
|Business Systems Specialists           |
|Consulting Members of Technical Staff  |
|Pentaho Developers                     |
|Technology Leads                       |
|Activation Managers                    |
|Application Support Specialists        |
|Statistical Data Analysts              |
|Trust Officers                         |
|Merchandise Analysts                   |
|Bulk Loaders                           |
|Studio Associates                      |
|Registered Dental Assistants           |
|Directors of Elementary                |
|EDI Developers                         |
|Business Unit Managers                 |
|Directors of Philanthropy              |
|ERP Functional Analysts          

In [98]:
# inspect the column
df.select("TITLE_CLEAN") \
  .distinct() \
  .show(50, truncate=False)

+-------------------------------------------------------------------------------------------+
|TITLE_CLEAN                                                                                |
+-------------------------------------------------------------------------------------------+
|entry level business management                                                            |
|oracle scm                                                                                 |
|oracle ebs r service and repair                                                            |
|information security ii data analytics                                                     |
|sap s fico resource                                                                        |
|senior sap sd otc functional                                                               |
|quality data entry a shift am pm                                                           |
|manager oracle scm                                         

In [101]:
# drop these columns after inspecting them
df = df.drop("TITLE_CLEAN", "LOT_CAREER_AREA_NAME","LOT_SPECIALIZED_OCCUPATION_NAME", "LOT_V6_SPECIALIZED_OCCUPATION_NAME", "LOT_V6_CAREER_AREA_NAME")

Reasons for dropping columns:

TITLE_CLEAN : Many entries contained overly verbose, unstructured, or non-standardized job titles that could not be reliably categorized.

LOT_SPECIALIZED_OCCUPATION_NAME: Too granular. 

LOT_CAREER_AREA_NAME: Majority of job postings were concentrated within a single category, “Information Technology and Computer Science,” which reduced its usefulness for segmentation and analysis.

LOT_V6_SPECIALIZED_OCCUPATION_NAME: Same labels as LOT_SPECIALIZED_OCCUPATION_NAME.

LOT_V6_CAREER_AREA_NAME: Same labels as LOT_CAREEER_AREA_NAME and also did not add variability for analysis.

In [134]:
df.groupBy("TITLE_NAME") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(100, truncate=False)

+------------------------------------+-----+
|TITLE_NAME                          |count|
+------------------------------------+-----+
|Data Analysts                       |8156 |
|Unclassified                        |2945 |
|Business Intelligence Analysts      |1992 |
|Enterprise Architects               |1891 |
|Oracle Cloud HCM Consultants        |924  |
|Data Modelers                       |652  |
|Data Governance Analysts            |557  |
|Data Analytics Engineers            |508  |
|Data Quality Analysts               |451  |
|Data Management Analysts            |427  |
|Solutions Architects                |424  |
|SAP Consultants                     |411  |
|Data and Reporting Analysts         |405  |
|ERP Business Analysts               |401  |
|SAP EWM Consultants                 |380  |
|Oracle Cloud Financials Consultants |375  |
|Principal Architects                |372  |
|Enterprise Solutions Architects     |350  |
|Data Analytics Interns              |348  |
|Lead Data

In [104]:
df.select("LOT_OCCUPATION_NAME").distinct().show(truncate=False)

+--------------------------------------------------------------------+
|LOT_OCCUPATION_NAME                                                 |
+--------------------------------------------------------------------+
|Business / Management Analyst                                       |
|Business Intelligence Analyst                                       |
|Market Research Analyst                                             |
|Computer Systems Engineer / Architect                               |
|Data / Data Mining Analyst                                          |
|Clinical Analyst / Clinical Documentation and Improvement Specialist|
+--------------------------------------------------------------------+



### **Skills**

Skill-related variables were processed independently to preserve their distinct meanings, including software, specialized, common skills, and certifications. After filtering the dataset to include only relevant data science and analytics roles, skill values were cleaned and transformed from string-based lists into individual observations. This enabled frequency-based analysis and the identification of the most in-demand skills across roles.

In [102]:
# view a smaller subset of selected columns
# specifically the skills related columns
df.select(
    "SKILLS_NAME",
    "SPECIALIZED_SKILLS_NAME",
    "SOFTWARE_SKILLS_NAME",
    "COMMON_SKILLS_NAME",
    "CERTIFICATIONS_NAME",
).show(20, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------

In [105]:
# filter for roles within our topic domain
from pyspark.sql.functions import lower, col

domain_df = df.filter(
    col("LOT_OCCUPATION_NAME").isin([
        "Data / Data Mining Analyst",
        "Business Intelligence Analyst",
        "Business / Management Analyst",
        "Computer Systems Engineer / Architect"
    ])
)

In [121]:
# select skills columns
skill_cols = [
    "SOFTWARE_SKILLS_NAME",      # technological skills
    "SPECIALIZED_SKILLS_NAME",   # technical/domain skills
    "COMMON_SKILLS_NAME",        # soft skills
    "CERTIFICATIONS_NAME"        # specialty skills
]

In [118]:
# clean and extract skills
from pyspark.sql.functions import regexp_replace, split, explode, trim, lower, col

# clean the string and remove unwanted characters such as list brackets, quotes, and new lines
def extract_skills(dataframe, skill_col):
    cleaned = dataframe.withColumn(
        "SKILL_CLEAN",
        regexp_replace(col(skill_col), r'[\[\]\n"]', "") 
    )

# explode so its one row per skill and rename column
    exploded = cleaned.select(
        explode(split(col("SKILL_CLEAN"), ",")).alias("SKILL")
    )

# clean formatting so everything is lower, trim any spaces, and remove empty values
    return exploded.withColumn(
        "SKILL",
        trim(lower(col("SKILL")))
    ).filter(col("SKILL") != "")

In [122]:
# create datasets per skill type
software_df = extract_skills(domain_df, "SOFTWARE_SKILLS_NAME")
specialized_df = extract_skills(domain_df, "SPECIALIZED_SKILLS_NAME")
common_df = extract_skills(domain_df, "COMMON_SKILLS_NAME")
certifications_df = extract_skills(domain_df, "CERTIFICATIONS_NAME")

In [123]:
# get top skills for software
software_top = software_df.groupBy("SKILL") \
    .count() \
    .orderBy("count", ascending=False)

software_top.show(40, truncate=False)

+------------------------------------------------+-----+
|SKILL                                           |count|
+------------------------------------------------+-----+
|sql (programming language)                      |20651|
|microsoft excel                                 |12178|
|python (programming language)                   |11613|
|sap applications                                |11551|
|dashboard                                       |11290|
|tableau (business intelligence software)        |11192|
|power bi                                        |10331|
|microsoft office                                |7172 |
|microsoft powerpoint                            |6787 |
|r (programming language)                        |5859 |
|microsoft azure                                 |4579 |
|amazon web services                             |4530 |
|oracle cloud                                    |3758 |
|sas (software)                                  |3383 |
|application programming interf

In [110]:
# get top skills for specialized
specialized_top = specialized_df.groupBy("SKILL") \
    .count() \
    .orderBy("count", ascending=False)

specialized_top.show(20, truncate=False)

+----------------------------------------+-----+
|SKILL                                   |count|
+----------------------------------------+-----+
|data analysis                           |26998|
|sql (programming language)              |20651|
|computer science                        |16321|
|project management                      |12955|
|business process                        |12304|
|business requirements                   |12129|
|finance                                 |11651|
|python (programming language)           |11613|
|sap applications                        |11551|
|dashboard                               |11290|
|tableau (business intelligence software)|11192|
|power bi                                |10331|
|business intelligence                   |9899 |
|agile methodology                       |8505 |
|data management                         |8276 |
|statistics                              |7998 |
|data modeling                           |7882 |
|data visualization 

In [111]:
# get top skills for common
common_top = common_df.groupBy("SKILL") \
    .count() \
    .orderBy("count", ascending=False)

common_top.show(20, truncate=False)

+---------------------------------+-----+
|SKILL                            |count|
+---------------------------------+-----+
|communication                    |30190|
|management                       |23414|
|problem solving                  |17630|
|leadership                       |16718|
|operations                       |13903|
|microsoft excel                  |12178|
|detail oriented                  |11507|
|presentations                    |11278|
|planning                         |10876|
|writing                          |10458|
|research                         |9579 |
|consulting                       |8854 |
|troubleshooting (problem solving)|8439 |
|innovation                       |8407 |
|information technology           |8297 |
|decision making                  |8257 |
|customer service                 |8145 |
|sales                            |7964 |
|microsoft office                 |7172 |
|analytical skills                |7158 |
+---------------------------------

In [116]:
certifications_top = certifications_df.groupBy("SKILL") \
    .count() \
    .orderBy("count", ascending=False)

certifications_top.show(20, truncate=False)

+-----------------------------------------------------------------+-----+
|SKILL                                                            |count|
+-----------------------------------------------------------------+-----+
|security clearance                                               |1574 |
|master of business administration (mba)                          |1255 |
|top secret-sensitive compartmented information (ts/sci clearance)|1151 |
|valid driver's license                                           |1090 |
|certified information systems security professional              |972  |
|secret clearance                                                 |961  |
|sap certification                                                |740  |
|project management professional certification                    |547  |
|certified information system auditor (cisa)                      |538  |
|oracle certification                                             |400  |
|certified information security manage

In [125]:
# limit top specializd skills to 15 for plotting
top_specialized = specialized_top.limit(15)

# convert it to pandas
top_specialized_pd = top_specialized.toPandas()

In [128]:
# create a plotly bar chart
import plotly.express as px

fig = px.bar(
    top_specialized_pd,
    x="count",
    y="SKILL",
    orientation="h",
    color="count",
    color_continuous_scale="Blues",
    title="<b>Top 10 Specialized Skills in Data Analytics & Data Science Roles</b>"
)

fig.update_layout(
    # Background styling
    template="plotly_white",
    paper_bgcolor="#f5f7fa",   # slightly darker than white
    plot_bgcolor="#f5f7fa",

    # Font styling
    font=dict(
        family="Inter, sans-serif",
        size=14,
        color="#2c3e50"
    ),

    # Title styling
    title=dict(
        x=0.5,
        font=dict(size=22)
    ),

    # Axis labels
    xaxis_title="<b>Number of Job Postings</b>",
    yaxis_title="<b>Skill</b>",

    # Order
    yaxis={'categoryorder':'total ascending'}
)

fig.update_traces(
    marker=dict(line=dict(width=1, color='black'))
)

fig.show()


In [129]:
# 
top_common = common_top.limit(10)
top_common_pd = top_common.toPandas()

In [146]:
top_common_pd = top_common_pd.sort_values("count", ascending=True)

fig = px.bar(
    top_common_pd,
    x="count",
    y="SKILL",
    orientation="h",
    color="count",
    color_discrete_sequence=["#3A86FF"],
    title="<b>Top 10 Common Soft Skills in Data Analytics, Data Science, and ML Roles</b>",
    labels={
        "SKILL": "<b>Skill</b>",
        "count": "<b>Number of Job Postings</b>"
    }
)

fig.update_layout(
    font=dict(family="Inter, Arial, sans-serif", size=14, color="#24324A"),
    title=dict(x=0.5, font=dict(size=18, color="#24324A")),
    plot_bgcolor="#E5ECF6",
    paper_bgcolor="white",
    coloraxis_showscale=False,
    bargap=0.28,
    margin=dict(l=180, r=50, t=80, b=60),
    xaxis=dict(
        gridcolor="white",
        zerolinecolor="white"
    ),
    yaxis=dict(
        showgrid=False
    )
)

fig.update_traces(
    marker_line_width=0,
    opacity=0.95,
    texttemplate="%{x}",
    textposition="outside"
)

fig.show()


    "#636EFA",  # blue
    "#EF553B",  # red
    "#00CC96",  # green
    "#AB63FA",  # purple
    "#FFA15A"   # orange

In [147]:
skills_by_role = domain_df.withColumn(
    "SKILL_CLEAN",
    regexp_replace(col("SOFTWARE_SKILLS_NAME"), r'[\[\]\n"]', "")
).select(
    "LOT_OCCUPATION_NAME",
    explode(split(col("SKILL_CLEAN"), ",")).alias("SKILL")
)
skills_by_role.show(20, truncate=False)

+-----------------------------+----------------------------+
|LOT_OCCUPATION_NAME          |SKILL                       |
+-----------------------------+----------------------------+
|Business Intelligence Analyst|                            |
|Business Intelligence Analyst|  SAP Logistics             |
|Business Intelligence Analyst|  SAP Sales And Distribution|
|Business Intelligence Analyst|  SAP Material Management   |
|Business / Management Analyst|                            |
|Business / Management Analyst|                            |
|Business / Management Analyst|                            |
|Business Intelligence Analyst|                            |
|Business Intelligence Analyst|                            |
|Data / Data Mining Analyst   |  Microsoft SharePoint      |
|Data / Data Mining Analyst   |  Microsoft Azure           |
|Data / Data Mining Analyst   |  Kubernetes                |
|Data / Data Mining Analyst   |  Powerapps                 |
|Data / Data Mining Anal

### **Job Description**

Missing values in the job description field (BODY) were replaced with empty strings. Since this variable represents unstructured text, imputing numerical values is not appropriate, and removing observations would result in unnecessary data loss. This approach preserves the dataset while allowing for potential text-based analysis without introducing bias.

In [148]:
# replace null values in body with empty string
# this avoids unecessary data loss
df = df.fillna({"BODY": ""})

### **Industry**

In [149]:
# view a smaller subset of selected columns
# specifically the industry related columns
df.select(
    "NAICS2_NAME",
    "NAICS_2022_2_NAME",
    "LIGHTCAST_SECTORS_NAME",
).show(20, truncate=False)

+---------------------+---------------------+-----------------------------------------------------+
|NAICS2_NAME          |NAICS_2022_2_NAME    |LIGHTCAST_SECTORS_NAME                               |
+---------------------+---------------------+-----------------------------------------------------+
|Unclassified Industry|Unclassified Industry|NULL                                                 |
|Unclassified Industry|Unclassified Industry|NULL                                                 |
|Unclassified Industry|Unclassified Industry|NULL                                                 |
|Unclassified Industry|Unclassified Industry|NULL                                                 |
|Unclassified Industry|Unclassified Industry|NULL                                                 |
|Unclassified Industry|Unclassified Industry|NULL                                                 |
|Unclassified Industry|Unclassified Industry|NULL                                                 |


In [150]:
# LIGHTCAST_SECTORS_NAME is being dropped because it has too many missing values
# NAICS_2022_2_NAME is being dropped because it has the same values as NAICS2_NAME
df = df.drop("LIGHTCAST_SECTORS_NAME", "NAICS_2022_2_NAME")

### **Experience & Education**

**Cleaning the MIN_YEARS_EXPERIENCE column**

Missing values in MIN_YEARS_EXPERIENCE were addressed using a hybrid approach. Internship postings were assigned zero experience based on domain knowledge, while remaining missing values were imputed using the median to preserve the distribution and reduce bias.

In [46]:
# internship postings are assingned zero experience
from pyspark.sql.functions import when, col
df = df.withColumn(
    "MIN_YEARS_EXPERIENCE_CLEANED",
    when(
        (col("MIN_YEARS_EXPERIENCE").isNull()) & (col("IS_INTERNSHIP") == True),
        0
    ).otherwise(col("MIN_YEARS_EXPERIENCE"))
)
# drop the helper column AFTER using it
df = df.drop("IS_INTERNSHIP")

# check remaining nulls
print("Remaining nulls:",
      df.filter(col("MIN_YEARS_EXPERIENCE_CLEANED").isNull()).count())

Remaining nulls: 6575


In [47]:
# inspecting whats left after mapping
# this helps us look for nulls, extreme values, and odd distrubtions
df.select("MIN_YEARS_EXPERIENCE_CLEANED").describe().show()

df.groupBy("MIN_YEARS_EXPERIENCE_CLEANED") \
  .count() \
  .orderBy("MIN_YEARS_EXPERIENCE_CLEANED") \
  .show(20)

+-------+----------------------------+
|summary|MIN_YEARS_EXPERIENCE_CLEANED|
+-------+----------------------------+
|  count|                       22661|
|   mean|           5.425312210405543|
| stddev|           3.378327955608337|
|    min|                           0|
|    max|                          15|
+-------+----------------------------+



+----------------------------+-----+
|MIN_YEARS_EXPERIENCE_CLEANED|count|
+----------------------------+-----+
|                        NULL| 6575|
|                           0|  584|
|                           1| 1283|
|                           2| 3060|
|                           3| 2810|
|                           4| 1553|
|                           5| 5057|
|                           6| 1303|
|                           7| 1314|
|                           8| 1581|
|                           9|  204|
|                          10| 1949|
|                          11|   78|
|                          12| 1470|
|                          13|   22|
|                          14|   56|
|                          15|  337|
+----------------------------+-----+



The distribution here looks reasonable.
Min = 0 → good (entry-level / internships)
Max = 15 → good (no extreme outliers)
Mean ≈ 5.4 → realistic
Most values between 2–6 years → makes sense for job postings

The minimum years of experience variable was successfully cleaned and validated. Following the imputation of missing values for internship roles, the remaining distribution of experience values was found to be reasonable, with values ranging from 0 to 15 years and a mean of approximately 5.4 years. Remaining missing values, which likely represent unspecified experience requirements, were removed to preserve data integrity and avoid introducing bias through imputation. This resulted in a clean and interpretable feature suitable for regression modeling.

In [48]:
# drop remaining nulls
df = df.dropna(subset=["MIN_YEARS_EXPERIENCE_CLEANED"])

In [ ]:
# adding a squared feature too help capture ???
from pyspark.sql.functions import col

df = df.withColumn(
    "MIN_YEARS_EXPERIENCE_SQ",
    col("MIN_YEARS_EXPERIENCE_CLEANED") ** 2
)

### **Salary**

The variables SALARY_FROM and SALARY_TO exhibited a high proportion of missing values, with over 55% of observations lacking complete salary range information. Due to this level of missingness, these variables were excluded from the analysis. Instead, the SALARY variable was retained, as it provides a standardized estimate of compensation and allows for more consistent modeling without substantial data loss.

In [44]:
# drop missing data in salary
# Rows with missing salary values (~57%) were removed to avoid introducing bias through imputation
df = df.dropna(subset=["SALARY"])

# drop salary_from and salary_to
df = df.drop("SALARY_FROM", "SALARY_TO")

### **Work Setup**

**Cleaning EMPLOYMENT_TYPE_NAME column**

The EMPLOYMENT_TYPE_NAME variable was cleaned and consolidated into a new feature, EMPLOYMENT_TYPE_NAME_CLEANED, by standardizing and grouping its distinct values into three clear categories: Part-time, Flexible, and Full-time, improving consistency and interpretability for modeling.

In [49]:
# Inspecting EMPLOYMENT_TYPE_NAME for distinct values
df.select("EMPLOYMENT_TYPE_NAME").distinct().show(100, truncate=False)

+------------------------+
|EMPLOYMENT_TYPE_NAME    |
+------------------------+
|Part-time / full-time   |
|Part-time (â‰¤ 32 hours)|
|Full-time (> 32 hours)  |
+------------------------+



In [51]:
from pyspark.sql.functions import when, col

# rename column categories 
df = df.withColumn(
    "EMPLOYMENT_TYPE_NAME_CLEANED",
    when(col("EMPLOYMENT_TYPE_NAME") == "Full-time (> 32 hours)", "Full-time")
    .when(col("EMPLOYMENT_TYPE_NAME") == "Part-time (â‰¤ 32 hours)", "Part-time")
    .when(col("EMPLOYMENT_TYPE_NAME") == "Part-time / full-time", "Flexible")
    .otherwise("other")
)

# Inspecting EMPLOYMENT_TYPE_NAME for distinct values
df.select("EMPLOYMENT_TYPE_NAME_CLEANED").distinct().show(100, truncate=False)

+----------------------------+
|EMPLOYMENT_TYPE_NAME_CLEANED|
+----------------------------+
|Part-time                   |
|Flexible                    |
|Full-time                   |
+----------------------------+



**Cleaning the REMOTE_TYPE_NAME column**

The REMOTE_TYPE_NAME variable was cleaned and transformed into a new feature, REMOTE_TYPE_NAME_CLEANED, by standardizing and grouping its distinct values into three consistent categories: Remote, Hybrid, and Onsite, improving clarity and making the variable more suitable for modeling. Additionally, any unknown values were assumed to be Hybrid, reflecting common post-COVID-19 pandemic workplace practices where many roles default to a hybrid structure. This improved both the clarity and practical relevance of the variable for modeling.

In [52]:
# Inspecting REMOTE_TYPE_NAME for distinct values
df.select("REMOTE_TYPE_NAME").distinct().show(100, truncate=False)

+----------------+
|REMOTE_TYPE_NAME|
+----------------+
|Remote          |
|[None]          |
|Not Remote      |
|Hybrid Remote   |
+----------------+



In [53]:
from pyspark.sql.functions import when, col, lower, trim

df= df.withColumn(
    "REMOTE_TYPE_NAME_CLEANED",
    trim(lower(col("REMOTE_TYPE_NAME")))
)
df = df.withColumn(
    "REMOTE_TYPE_NAME_CLEANED",
    when(col("REMOTE_TYPE_NAME_CLEANED") == "remote", "remote")
    .when(col("REMOTE_TYPE_NAME_CLEANED") == "hybrid remote", "hybrid")
    .when(col("REMOTE_TYPE_NAME_CLEANED") == "not remote", "onsite")
    .when(col("REMOTE_TYPE_NAME_CLEANED") == "[none]", "hybrid")
    .when(col("REMOTE_TYPE_NAME_CLEANED").isNull(), "hybrid")
    .otherwise("other")
)

#verify categories
df.groupBy("REMOTE_TYPE_NAME_CLEANED").count().show()

+------------------------+-----+
|REMOTE_TYPE_NAME_CLEANED|count|
+------------------------+-----+
|                  onsite|  432|
|                  hybrid|17223|
|                  remote| 5006|
+------------------------+-----+

